# Notebook 02_01 — Feature Selection: Correlation

Runs **Correlation** across K = {4, 8, 16} × 6 classifiers × 5 seeds.
Uses the pre-computed ranking from `02_00_fs_rankings.ipynb` (`models/fs_rankings.pkl`).

**Results saved incrementally to** `results/fs_correlation_raw.csv` — if this notebook crashes mid-run, just re-run it: completed (K, seed, classifier) combinations are skipped automatically.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)
Imports OK


In [2]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']
FEATURE_NAMES = d['feature_names']

with open(f'{MODELS_DIR}fs_rankings.pkl', 'rb') as f:
    RANKINGS = pickle.load(f)

METHOD_NAME = 'Correlation'
ranked_indices = RANKINGS[METHOD_NAME]

print(f'Method: {METHOD_NAME}')
print('Top-16 features:', [FEATURE_NAMES[i] for i in ranked_indices[:16]])

Method: Correlation
Top-16 features: ['ip.version', 'ip.proto', 'ip.ttl', 'arp.opcode', 'frame.encap_type', 'wlan.fc.subtype', 'wlan_radio.frequency', 'radiotap.channel.freq', 'wlan_radio.channel', 'wlan_radio.phy', 'radiotap.length', 'wlan.fc.type', 'wlan_radio.signal_dbm', 'tcp.flags.ack', 'radiotap.channel.flags.ofdm', 'tcp.srcport']


## Transform function (slice to top-K features by this method's ranking)

In [3]:
def transform_fn(K):
    selected = ranked_indices[:K]
    return X_train[:, selected], X_val[:, selected], X_test[:, selected], K

## Run experiment grid (resumable)

In [4]:
RESULTS_CSV = f'{RESULTS_DIR}fs_correlation_raw.csv'

run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

[Correlation] K=4 seed=42 DT     F1=0.3181  train=0.0s  inf=0.0001ms/sample  (1/90)
[Correlation] K=4 seed=42 RF     F1=0.3182  train=1.0s  inf=0.0034ms/sample  (2/90)
[Correlation] K=4 seed=42 XGB    F1=0.3181  train=1.4s  inf=0.0008ms/sample  (3/90)
[Correlation] K=4 seed=42 LGBM   F1=0.3182  train=0.7s  inf=0.0012ms/sample  (4/90)
[Correlation] K=4 seed=42 KNN    F1=0.1036  train=0.1s  inf=0.5424ms/sample  (5/90)
[Correlation] K=4 seed=42 MLP    F1=0.3181  train=16.3s  inf=0.0011ms/sample  (6/90)
[Correlation] K=4 seed=52 DT     F1=0.3181  train=0.0s  inf=0.0001ms/sample  (7/90)
[Correlation] K=4 seed=52 RF     F1=0.3181  train=1.0s  inf=0.0028ms/sample  (8/90)
[Correlation] K=4 seed=52 XGB    F1=0.3181  train=1.6s  inf=0.0009ms/sample  (9/90)
[Correlation] K=4 seed=52 LGBM   F1=0.3182  train=0.8s  inf=0.0012ms/sample  (10/90)
[Correlation] K=4 seed=52 KNN    F1=0.1057  train=0.1s  inf=0.4399ms/sample  (11/90)
[Correlation] K=4 seed=52 MLP    F1=0.4076  train=34.8s  inf=0.0013ms/sam

## Quick summary

In [5]:
res_df = pd.read_csv(RESULTS_CSV)
summary = res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std'])
print(summary.to_string())

                     f1               train_time_s           inference_ms          
                   mean           std         mean       std         mean       std
K  classifier                                                                      
4  DT          0.318057  3.086904e-07     0.016295  0.001011     0.000055  0.000004
   KNN         0.104846  1.123438e-03     0.076992  0.002970     0.492857  0.042011
   LGBM        0.318146  8.109363e-05     0.687034  0.060847     0.001220  0.000019
   MLP         0.353728  4.913526e-02    19.847926  9.002256     0.001072  0.000433
   RF          0.318116  8.114512e-05     1.028261  0.008278     0.003001  0.000321
   XGB         0.318057  2.780231e-07     1.360422  0.168120     0.000824  0.000035
8  DT          0.631067  6.199769e-07     0.028727  0.000481     0.000061  0.000002
   KNN         0.249040  5.485355e-02     0.139343  0.003192     0.265584  0.079448
   LGBM        0.631049  2.407755e-06     0.969181  0.020255     0.001922  0